# Low-Entropy Tool Design for Reliable AI Agents

## What you will build

You will build the tools behind an agent on a customer account portal. Customers ask about their
plan, their invoices and their contact email, and they report problems that need a person. The model
reads a list of tools on every request and picks the one it thinks fits, then your code runs it.

Over the years the portal grew 25 generic tools with vague names and loose arguments, and the model
now has to guess which one a request belongs to. The diagram shows the three mistakes this course
stops: the wrong tool answering the question, a guessed customer id reaching the portal, and a tool
running on arguments that nobody checked.

![What you will build](images/portal-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. Without an API key it replays
responses recorded from real runs, so you can follow the whole course for free, and with a key it
calls the model live.

In [1]:
import copy
import json
import math
import re
from collections import Counter

from vault import get_client, load_env, model_for

load_env()
client = get_client("10-low-entropy-tool-design/01-design-low-entropy-tools")
MODEL = model_for("default")

print(f"Client ready. Every request in this notebook uses {MODEL}.")

Client ready. Every request in this notebook uses google/gemini-2.5-flash-lite.


## Step 1: Create the account portal backend

The agent needs a real portal to act on, so we start with two customers and their invoices. Every
customer id has the same shape, `CUST-` and five digits, and the portal has four jobs it can do.

In [2]:
CUSTOMERS = {
    "CUST-10482": {"company": "Northwind", "plan": "Team", "seats": 25,
                   "contact_email": "admin@northwind.example"},
    "CUST-20931": {"company": "Harbor Labs", "plan": "Business", "seats": 80,
                   "contact_email": "it@harbor.example"},
}
INVOICES = [
    {"customer_id": "CUST-10482", "invoice": "INV-7001", "amount_cents": 45000, "status": "paid"},
    {"customer_id": "CUST-10482", "invoice": "INV-7002", "amount_cents": 45000, "status": "open"},
    {"customer_id": "CUST-20931", "invoice": "INV-7003", "amount_cents": 144000, "status": "open"},
]
SUPPORT_CASES = []   # every case the portal has opened for the support team


def get_account_profile(customer_id):
    """Return the plan, seats and contact email for one customer."""
    if customer_id not in CUSTOMERS:
        return {"error": f"No customer with id {customer_id}"}
    return {"customer_id": customer_id, **CUSTOMERS[customer_id]}


print(get_account_profile("CUST-10482"))

{'customer_id': 'CUST-10482', 'company': 'Northwind', 'plan': 'Team', 'seats': 25, 'contact_email': 'admin@northwind.example'}


The other three jobs read invoices, change the contact email and open a case for a person. Two of
them write to the portal, so a wrong call changes a real account.

In [3]:
def list_invoices(customer_id, status="all"):
    """Return one customer's invoices, filtered to open, paid or all."""
    return [invoice for invoice in INVOICES if invoice["customer_id"] == customer_id
            and status in ("all", invoice["status"])]


def update_contact_email(customer_id, new_email):
    """Change where the portal sends account email. This writes to the account."""
    CUSTOMERS[customer_id]["contact_email"] = new_email
    return {"customer_id": customer_id, "contact_email": new_email}


def open_support_case(customer_id, topic, summary):
    """Open a case for a person on the support team to pick up."""
    SUPPORT_CASES.append({"customer_id": customer_id, "topic": topic, "summary": summary})
    return {"case_number": len(SUPPORT_CASES), "topic": topic}


print(list_invoices("CUST-20931", status="open"))

[{'customer_id': 'CUST-20931', 'invoice': 'INV-7003', 'amount_cents': 144000, 'status': 'open'}]


## Step 2: Register the 25 generic tools the portal grew

We start from the tool suite the portal really has, because that is what most teams inherit. Each
project added tools and none removed any, so there are now 25, each with a vague sentence and a
**schema**, which is the written shape of the allowed arguments, holding one loose `id`.

![Register the 25 generic tools the portal grew](images/tool-selection-step-1.svg)

In [4]:
GENERIC_TOOL_NAMES = [
    "get_customer", "get_customer_info", "get_account", "get_account_details",
    "lookup_account", "fetch_profile", "get_user", "search_customers",
    "get_billing", "get_billing_info", "get_invoices", "list_invoices", "get_invoice",
    "get_payments", "get_balance", "get_plan", "get_subscription",
    "update_customer", "update_account", "update_email", "change_contact",
    "set_preferences", "create_ticket", "open_case", "contact_support",
]


def describe_generic_tool(name):
    """Describe a tool the way the portal grew them: a vague sentence and one loose id."""
    sentence = name.replace("_", " ").capitalize() + "."
    return {"type": "function", "function": {
        "name": name, "description": sentence,
        "parameters": {"type": "object", "properties": {"id": {"type": "string"}}}}}


GENERIC_TOOLS = [describe_generic_tool(name) for name in GENERIC_TOOL_NAMES]
print(f"{len(GENERIC_TOOLS)} generic tools, for example {GENERIC_TOOLS[2]['function']}")

25 generic tools, for example {'name': 'get_account', 'description': 'Get account.', 'parameters': {'type': 'object', 'properties': {'id': {'type': 'string'}}}}


Only four of those names are connected to the live portal, one for each job. The others were added
by earlier projects and read from old copies of the data, so a request that lands on them gets a
stale answer.

In [5]:
WIRED_GENERIC_TOOLS = {"profile": "get_account", "invoices": "list_invoices",
                       "email": "update_email", "case": "create_ticket"}

print(f"wired to the live portal: {sorted(WIRED_GENERIC_TOOLS.values())}")
print(f"left behind by earlier projects: {len(GENERIC_TOOLS) - len(WIRED_GENERIC_TOOLS)} tools")

wired to the live portal: ['create_ticket', 'get_account', 'list_invoices', 'update_email']
left behind by earlier projects: 21 tools


## Step 3: Measure tool selection accuracy with real runs

We measure before we change anything, so every later fix has a number to beat. The test set holds
four requests for each job, phrased the way different customers would write them, and each one
names a real customer id.

![Measure tool selection accuracy with real runs](images/tool-selection-step-2.svg)

In [6]:
TEST_REQUESTS = [
    ("profile", "What plan is CUST-10482 on right now?"),
    ("profile", "Who is the contact person on file for CUST-20931?"),
    ("profile", "How many seats does CUST-10482 have?"),
    ("profile", "Pull up the account details for CUST-20931."),
    ("invoices", "Which invoices are still unpaid on CUST-10482?"),
    ("invoices", "How much does CUST-20931 owe us at the moment?"),
    ("invoices", "Did CUST-10482 pay last month's bill?"),
    ("invoices", "Send me the billing history for CUST-20931."),
    ("email", "Please change the contact email on CUST-10482 to ops@northwind.example."),
    ("email", "Our admin left. Make billing@harbor.example the contact for CUST-20931."),
    ("email", "Update CUST-10482 so account emails go to it@northwind.example."),
    ("email", "Swap the email on file for CUST-20931 to admin@harbor.example."),
    ("case", "I can't log in to the portal on account CUST-20931. Can someone look at it?"),
    ("case", "The export button crashes for CUST-10482. Please raise it with engineering."),
    ("case", "CUST-20931 was charged twice this month and we need a person to sort it out."),
    ("case", "Nothing loads on the dashboard for CUST-10482, please escalate."),
]
print(f"{len(TEST_REQUESTS)} test requests across {len({job for job, _ in TEST_REQUESTS})} jobs")

16 test requests across 4 jobs


`pick_tools` sends one request and returns each **tool call**, which is the model asking your code to
run a named function, sent as data. Nothing runs yet, because this step only measures what the model
chose.

In [7]:
SYSTEM_PROMPT = ("You are the account portal assistant for a software company. "
                 "Use your tools to answer.")


def pick_tools(tools, request):
    """Send one request and return every tool call as a name and its arguments."""
    response = client.chat.completions.create(
        model=MODEL, max_tokens=400, tools=tools,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": request}])
    tool_calls = response.choices[0].message.tool_calls or []
    return [(call.function.name, json.loads(call.function.arguments)) for call in tool_calls]


print(pick_tools(GENERIC_TOOLS, "What plan is CUST-10482 on right now?"))

[('get_plan', {'id': 'CUST-10482'})]


A request counts as correct only when the model made exactly one call and that call went to the tool
wired for its job. `run_selection_eval` applies that rule to every request in the test set.

In [8]:
def run_selection_eval(tools, wired_tools):
    """Send every test request once and record which tools the model picked."""
    rows = []
    for job, request in TEST_REQUESTS:
        calls = pick_tools(tools, request)
        picked = [name for name, _ in calls]
        rows.append({"job": job, "picked": picked, "calls": calls,
                     "correct": picked == [wired_tools[job]]})
    return rows


def print_accuracy(label, rows):
    """Print how many requests went to the wired tool and nowhere else."""
    correct = sum(row["correct"] for row in rows)
    print(f"{label}: {correct}/{len(rows)} requests went to the wired tool "
          f"({correct / len(rows):.0%})")

Now the whole test set goes through the 25 generic tools, one request at a time.

In [9]:
generic_rows = run_selection_eval(GENERIC_TOOLS, WIRED_GENERIC_TOOLS)

for row in generic_rows:
    verdict = "right" if row["correct"] else "WRONG"
    print(f"{verdict:5}  {row['job']:8}  picked {row['picked'] or 'no tool'}")
print()
print_accuracy("25 generic tools", generic_rows)

WRONG  profile   picked ['get_plan']
WRONG  profile   picked ['get_customer']
WRONG  profile   picked no tool
WRONG  profile   picked ['get_account_details']
WRONG  invoices  picked no tool
WRONG  invoices  picked ['get_balance']
WRONG  invoices  picked ['get_payments']
WRONG  invoices  picked ['get_billing_info']
right  email     picked ['update_email']
WRONG  email     picked ['change_contact']
WRONG  email     picked ['update_account']
right  email     picked ['update_email']
right  case      picked ['create_ticket']
right  case      picked ['create_ticket']
right  case      picked ['create_ticket']
right  case      picked ['create_ticket']

25 generic tools: 6/16 requests went to the wired tool (38%)


Six of the sixteen requests reached the wired tool, and four of those six were support cases. Every
wrong pick went to a tool whose name fits the request as well as the wired one does, such as
`get_plan` for a plan question or `get_balance` for money owed. Two requests got no tool call at all,
and nothing in the 25 names tells the model which tool is connected to the live portal.

## Step 4: Measure selection entropy in bits

Accuracy says how often the model was right, but not how scattered its choices were, so we measure
that as well. **Selection entropy** is how spread out the model's choice of tool is for one job,
measured in bits, and it is zero when every request for that job lands on the same tool.

For one job, count where the picks landed and take the share `p` of each tool. The entropy is
`H = -sum(p * log2(p))`. Two tools picked equally often give 1 bit, four give 2 bits, and a suite of
25 tools could spread one choice across as many as `log2(25)` bits.

In [10]:
def measure_selection_entropy(rows, job):
    """Bits of spread in which tool the model picked for one job, and the pick counts."""
    picks = Counter(name for row in rows if row["job"] == job
                    for name in row["picked"] or ["no tool"])
    total = sum(picks.values())
    bits = -sum(count / total * math.log2(count / total) for count in picks.values())
    return abs(bits), dict(picks)   # abs() turns -0.0 into 0.0


def print_entropy(label, rows, tools):
    """Print the spread for each job, and the most spread a suite this size allows."""
    print(f"{label}: at most {math.log2(len(tools)):.2f} bits for {len(tools)} tools")
    for job in WIRED_GENERIC_TOOLS:
        bits, picks = measure_selection_entropy(rows, job)
        print(f"  {job:8} {bits:.2f} bits  {picks}")


print_entropy("25 generic tools", generic_rows, GENERIC_TOOLS)

25 generic tools: at most 4.64 bits for 25 tools
  profile  2.00 bits  {'get_plan': 1, 'get_customer': 1, 'no tool': 1, 'get_account_details': 1}
  invoices 2.00 bits  {'no tool': 1, 'get_balance': 1, 'get_payments': 1, 'get_billing_info': 1}
  email    1.50 bits  {'update_email': 2, 'change_contact': 1, 'update_account': 1}
  case     0.00 bits  {'create_ticket': 4}


Three of the four jobs spread across several tools. The profile and invoice requests each landed on
four different outcomes, which is 2.00 bits, and the email requests landed on three. Only the case
job stayed at zero bits, because `create_ticket` was the one name that matched all four ways of
reporting a problem. A high number here means the tool a customer reaches depends on how they
phrased the request, not on what they needed.

## Step 5: Replace them with four scoped tools

The fix for scattered choices is fewer tools that do not overlap, rather than better names for 25 of
them. We register four scoped tools, one for each job, and every choice inside a job, such as which
invoices to show, becomes a typed argument instead of a sibling tool.

![Replace them with four scoped tools](images/tool-selection-step-3.svg)

In [11]:
CUSTOMER_ID = {"type": "string"}
TOOL_JOBS = {
    "get_account_profile": ("Return the plan, seats and contact email for one customer.",
                            {"customer_id": CUSTOMER_ID}),
    "list_invoices": ("Return one customer's invoices, filtered by status.",
                      {"customer_id": CUSTOMER_ID,
                       "status": {"type": "string", "enum": ["open", "paid", "all"]}}),
    "update_contact_email": ("Change the contact email on one customer's account.",
                             {"customer_id": CUSTOMER_ID, "new_email": {"type": "string"}}),
    "open_support_case": ("Open a case for the support team about one customer.",
                          {"customer_id": CUSTOMER_ID, "summary": {"type": "string"},
                           "topic": {"type": "string",
                                     "enum": ["billing", "access", "bug", "other"]}}),
}


def describe_scoped_tool(name, description, properties):
    """Describe one tool with a single job, where every argument is typed and required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties,
                       "required": sorted(properties)}}}

Each scoped tool maps to one backend function, so the tool the model picks is always the one that
runs.

In [12]:
SCOPED_TOOLS = [describe_scoped_tool(name, text, properties)
                for name, (text, properties) in TOOL_JOBS.items()]
TOOL_REGISTRY = {"get_account_profile": get_account_profile, "list_invoices": list_invoices,
                 "update_contact_email": update_contact_email,
                 "open_support_case": open_support_case}
WIRED_SCOPED_TOOLS = {"profile": "get_account_profile", "invoices": "list_invoices",
                      "email": "update_contact_email", "case": "open_support_case"}

print(f"{len(SCOPED_TOOLS)} scoped tools: {list(TOOL_REGISTRY)}")

4 scoped tools: ['get_account_profile', 'list_invoices', 'update_contact_email', 'open_support_case']


The same sixteen requests go through the scoped suite, scored by the same eval.

In [13]:
scoped_rows = run_selection_eval(SCOPED_TOOLS, WIRED_SCOPED_TOOLS)

for row in scoped_rows:
    if not row["correct"]:
        print(f"WRONG  {row['job']:8}  picked {row['picked'] or 'no tool'}")
print_accuracy("25 generic tools", generic_rows)
print_accuracy("4 scoped tools  ", scoped_rows)
print()
print_entropy("4 scoped tools", scoped_rows, SCOPED_TOOLS)

WRONG  invoices  picked no tool
WRONG  case      picked no tool
WRONG  case      picked no tool
25 generic tools: 6/16 requests went to the wired tool (38%)
4 scoped tools  : 13/16 requests went to the wired tool (81%)

4 scoped tools: at most 2.00 bits for 4 tools
  profile  0.00 bits  {'get_account_profile': 4}
  invoices 0.81 bits  {'list_invoices': 3, 'no tool': 1}
  email    0.00 bits  {'update_contact_email': 4}
  case     1.00 bits  {'open_support_case': 2, 'no tool': 2}


Accuracy went from 6 of 16 to 13 of 16, and the spread fell to zero bits for the profile and email
jobs. None of the three misses went to a wrong tool, because each one was a question back to the
customer. The billing history request was asked which invoice status to show, and two case requests
were asked for a topic, because `topic` is a required argument that the customer never named.

| Suite | Requests that reached the wired tool | Bits for profile, invoices, email, case |
|---|---|---|
| 25 generic tools | 6 of 16 | 2.00, 2.00, 1.50, 0.00 |
| 4 scoped tools | 13 of 16 | 0.00, 0.81, 0.00, 1.00 |

The spread that remains for invoices and cases is all `no tool`, and a question back to the customer
is a far cheaper mistake than a stale answer from a tool nobody maintains.

## Step 6: Write activation criteria as positive conditions

A description also tells the model when a tool should not be called, and most teams write that as a
prohibition. We compare that with **positive activation framing**, a description that states the
conditions under which the tool should be called and what to do when they are not met.

![Write activation criteria as positive conditions](images/argument-checks-step-1.svg)

In [14]:
PROHIBITION = " Do not call this without a customer id."
ACTIVATION = (" Call this only when the customer has given an id matching CUST-XXXXX,"
              " such as CUST-10482. If they have not, ask them for it.")


def add_to_descriptions(tools, sentence):
    """Copy a tool suite with one sentence added to the end of every description."""
    framed = copy.deepcopy(tools)
    for tool in framed:
        tool["function"]["description"] += sentence
    return framed


PROHIBITION_TOOLS = add_to_descriptions(SCOPED_TOOLS, PROHIBITION)
ACTIVATION_TOOLS = add_to_descriptions(SCOPED_TOOLS, ACTIVATION)
print(ACTIVATION_TOOLS[1]["function"]["description"])

Return one customer's invoices, filtered by status. Call this only when the customer has given an id matching CUST-XXXXX, such as CUST-10482. If they have not, ask them for it.


Three requests test the framing, and none of them carries a usable customer id. The first gives a
company name, the second gives the number without `CUST-`, and the third gives the id in lower case
with an email address that has no domain ending.

In [15]:
ACTIVATION_REQUESTS = [
    "Hi, this is Priya from Northwind. What plan are we on?",
    "My customer number is 10482. Which invoices are unpaid?",
    "Change the email on cust-10482 to ops@northwind please.",
]


def run_framing_trial(tools, repeats=3):
    """Send each activation request a few times and collect every tool call."""
    return [call for request in ACTIVATION_REQUESTS for _ in range(repeats)
            for call in pick_tools(tools, request)]


def print_framing_trial(label, calls):
    """Print how many calls came back and the arguments each one carried."""
    print(f"{label}: {len(calls)} tool calls from {3 * len(ACTIVATION_REQUESTS)} requests")
    for name, arguments in calls:
        print(f"  {name}({arguments})")

Each suite gets the same nine requests, and only the description sentence differs between them.

In [16]:
prohibition_calls = run_framing_trial(PROHIBITION_TOOLS)
activation_calls = run_framing_trial(ACTIVATION_TOOLS)

print_framing_trial("prohibition", prohibition_calls)
print_framing_trial("positive activation", activation_calls)

prohibition: 7 tool calls from 9 requests
  get_account_profile({'customer_id': 'Northwind'})
  list_invoices({'status': 'open', 'customer_id': '10482'})
  list_invoices({'status': 'open', 'customer_id': '10482'})
  list_invoices({'status': 'open', 'customer_id': '10482'})
  update_contact_email({'customer_id': 'cust-10482', 'new_email': 'ops@northwind'})
  update_contact_email({'customer_id': 'cust-10482', 'new_email': 'ops@northwind'})
  update_contact_email({'customer_id': 'cust-10482', 'new_email': 'ops@northwind'})
positive activation: 6 tool calls from 9 requests
  list_invoices({'status': 'open', 'customer_id': 'CUST-10482'})
  list_invoices({'customer_id': 'CUST-10482', 'status': 'open'})
  list_invoices({'status': 'open', 'customer_id': 'CUST-10482'})
  update_contact_email({'new_email': 'ops@northwind', 'customer_id': 'cust-10482'})
  update_contact_email({'customer_id': 'cust-10482', 'new_email': 'ops@northwind'})
  update_contact_email({'new_email': 'ops@northwind', 'custom

The two framings differed on the plan question. With the prohibition, the model looked the account up
once with `Northwind` as the customer id, while the positive framing asked for the id in all three
attempts. Neither framing stopped the other two requests. The prohibition sent `10482` three times,
and the positive framing turned it into `CUST-10482` three times, which is a guessed id in the right
shape. Both suites sent the lower case id and the broken email address in all three attempts,
because an id was present, only in the wrong form.

## Step 7: Lock each schema with patterns and no extra fields

The description is only words, so the next move puts the rules into the schema itself. **Strict
schema locking** means setting `additionalProperties` to false, so no undeclared field is allowed,
and giving every id and email a `pattern` that its value has to match.

![Lock each schema with patterns and no extra fields](images/argument-checks-step-2.svg)

In [17]:
CUSTOMER_ID_PATTERN = "^CUST-[0-9]{5}$"
EMAIL_PATTERN = "^[^@ ]+@[^@ ]+[.][a-z]+$"
FIELD_PATTERNS = {"customer_id": CUSTOMER_ID_PATTERN, "new_email": EMAIL_PATTERN}


def lock_schema(tool):
    """Copy a tool, forbid undeclared fields and pin every id and email to a pattern."""
    locked = copy.deepcopy(tool)
    parameters = locked["function"]["parameters"]
    parameters["additionalProperties"] = False
    for field, rule in parameters["properties"].items():
        if field in FIELD_PATTERNS:
            rule["pattern"] = FIELD_PATTERNS[field]
    return locked


LOCKED_TOOLS = [lock_schema(tool) for tool in ACTIVATION_TOOLS]
print(json.dumps(LOCKED_TOOLS[2]["function"]["parameters"], indent=2))

{
  "type": "object",
  "properties": {
    "customer_id": {
      "type": "string",
      "pattern": "^CUST-[0-9]{5}$"
    },
    "new_email": {
      "type": "string",
      "pattern": "^[^@ ]+@[^@ ]+[.][a-z]+$"
    }
  },
  "required": [
    "customer_id",
    "new_email"
  ],
  "additionalProperties": false
}


The loose schemas from Step 2 let the argument names drift as well, because they declared only
`id`. The next cell lists the argument names the generic suite sent for the email job.

In [18]:
email_argument_names = Counter(tuple(sorted(arguments)) for row in generic_rows
                               if row["job"] == "email" for _, arguments in row["calls"])
print(f"argument names sent to the generic email tools: {dict(email_argument_names)}")

argument names sent to the generic email tools: {('email', 'id'): 1, ('id',): 1, ('id', 'prefs'): 1, ('id', 'new_email'): 1}


`find_pattern_breaks` lists every call whose id or email fails its pattern. The locked suite then
gets the same nine requests as before.

In [19]:
def find_pattern_breaks(calls):
    """Return every call whose id or email does not match its pattern."""
    return [(name, arguments) for name, arguments in calls
            if any(field in arguments and not re.fullmatch(pattern, str(arguments[field]))
                   for field, pattern in FIELD_PATTERNS.items())]


locked_calls = run_framing_trial(LOCKED_TOOLS)
print_framing_trial("locked schemas", locked_calls)
print()
for label, calls in (("prohibition", prohibition_calls),
                     ("positive activation", activation_calls), ("locked", locked_calls)):
    print(f"{label:20} {len(find_pattern_breaks(calls))} calls break a pattern")

locked schemas: 2 tool calls from 9 requests
  list_invoices({'customer_id': 'CUST-10482', 'status': 'open'})
  list_invoices({'status': 'open', 'customer_id': 'CUST-10482'})

prohibition          7 calls break a pattern
positive activation  3 calls break a pattern
locked               0 calls break a pattern


The loose email tools received four different sets of argument names across four requests, including
a `prefs` field that no schema declared, and one call carried the id with no email address at all.
With the locked schemas, the model sent no call for the plan question or the email change, and no
call broke a pattern. It still sent `CUST-10482` twice for a customer who only typed `10482`. That
guess matches the pattern, so a pattern checks the shape of a value but never whether the customer
gave it.

| Suite | Tool calls from 9 requests | Calls that break a pattern |
|---|---|---|
| Prohibition | 7 | 7 |
| Positive activation | 6 | 3 |
| Positive activation and locked schemas | 2 | 0 |

The locked schema still reaches the model as text in the request, and the tool call that comes back
is the model's own output. Nothing guarantees that the next call obeys the schema, so the next step
checks every call in code.

## Step 8: Check arguments in middleware before a tool runs

The schema changes what the model tends to send, but only your own code can refuse a call. We add
**middleware**, which is code that runs between two steps and can inspect or block what passes, so
every tool call is checked against its locked schema before the tool runs.

![Check arguments in middleware before a tool runs](images/argument-checks-step-3.svg)

In [20]:
LOCKED_SCHEMAS = {tool["function"]["name"]: tool["function"]["parameters"]
                  for tool in LOCKED_TOOLS}


def check_arguments(name, arguments):
    """Return every problem with one tool call. An empty list means it may run."""
    if name not in LOCKED_SCHEMAS:
        return [f"unknown tool {name!r}, the tools are {sorted(LOCKED_SCHEMAS)}"]
    schema = LOCKED_SCHEMAS[name]
    problems = [f"missing field {field!r}" for field in schema["required"]
                if field not in arguments]
    problems += [f"unexpected field {field!r}" for field in arguments
                 if field not in schema["properties"]]
    for field, value in arguments.items():
        rule = schema["properties"].get(field, {})
        if "pattern" in rule and not re.fullmatch(rule["pattern"], str(value)):
            problems.append(f"{field} {value!r} does not match {rule['pattern']}")
        if "enum" in rule and value not in rule["enum"]:
            problems.append(f"{field} {value!r} is not one of {rule['enum']}")
    if not problems and arguments["customer_id"] not in CUSTOMERS:
        problems.append(f"no customer with id {arguments['customer_id']!r}")
    return problems

Every call recorded in the last two steps now goes through `check_arguments`, to see which of them
the middleware would have stopped.

In [21]:
every_call = prohibition_calls + activation_calls + locked_calls
refused = [(name, arguments, check_arguments(name, arguments))
           for name, arguments in every_call if check_arguments(name, arguments)]

print(f"calls recorded: {len(every_call)}, refused by the middleware: {len(refused)}")
for name, arguments, problems in refused:
    print(f"  {name}({arguments})")
    print(f"    {problems}")

calls recorded: 15, refused by the middleware: 10
  get_account_profile({'customer_id': 'Northwind'})
    ["customer_id 'Northwind' does not match ^CUST-[0-9]{5}$"]
  list_invoices({'status': 'open', 'customer_id': '10482'})
    ["customer_id '10482' does not match ^CUST-[0-9]{5}$"]
  list_invoices({'status': 'open', 'customer_id': '10482'})
    ["customer_id '10482' does not match ^CUST-[0-9]{5}$"]
  list_invoices({'status': 'open', 'customer_id': '10482'})
    ["customer_id '10482' does not match ^CUST-[0-9]{5}$"]
  update_contact_email({'customer_id': 'cust-10482', 'new_email': 'ops@northwind'})
    ["customer_id 'cust-10482' does not match ^CUST-[0-9]{5}$", "new_email 'ops@northwind' does not match ^[^@ ]+@[^@ ]+[.][a-z]+$"]
  update_contact_email({'customer_id': 'cust-10482', 'new_email': 'ops@northwind'})
    ["customer_id 'cust-10482' does not match ^CUST-[0-9]{5}$", "new_email 'ops@northwind' does not match ^[^@ ]+@[^@ ]+[.][a-z]+$"]
  update_contact_email({'customer_id': 'cust

The five calls that passed all carried `CUST-10482`, the id the model completed from `10482`, because a check on shape and on whether the customer exists cannot tell a guess from an id the customer typed.

In the agent, a refused call must not crash the loop or disappear silently. `run_tool_with_middleware`
returns the problems as the tool result, together with what the model should do next, so the model
can recover on its next turn.

In [22]:
def run_tool_with_middleware(name, raw_arguments):
    """Check one tool call before it runs. A refused call returns its problems instead."""
    try:
        arguments = json.loads(raw_arguments)
    except json.JSONDecodeError:
        return {"error": f"arguments are not valid JSON: {raw_arguments!r}"}
    problems = check_arguments(name, arguments)
    if problems:
        return {"error": problems,
                "next_step": "Ask the customer for the correct value, then call the tool again."}
    return TOOL_REGISTRY[name](**arguments)


print(run_tool_with_middleware("list_invoices", '{"customer_id": "10482", "status": "open"}'))
print(run_tool_with_middleware("list_invoices", '{"customer_id": "CUST-10482", "status": "open"}'))

{'error': ["customer_id '10482' does not match ^CUST-[0-9]{5}$"], 'next_step': 'Ask the customer for the correct value, then call the tool again.'}
[{'customer_id': 'CUST-10482', 'invoice': 'INV-7002', 'amount_cents': 45000, 'status': 'open'}]


`run_agent` is a small agent loop that sends every tool call through the middleware. We give it the
weakest suite, the one with prohibitions and loose schemas, because that is where bad arguments came
back most often.

In [23]:
MAX_TURNS = 4   # the loop always ends, even if the model keeps asking for tools


def run_agent(request, tools):
    """Call the model, check each tool call in middleware, and repeat until it answers."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": request}]
    for turn in range(1, MAX_TURNS + 1):
        response = client.chat.completions.create(model=MODEL, max_tokens=400,
                                                  tools=tools, messages=messages)
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))
        if not message.tool_calls:
            return message.content
        for call in message.tool_calls:
            output = run_tool_with_middleware(call.function.name, call.function.arguments)
            print(f"turn {turn}: {call.function.name}({call.function.arguments})")
            print(f"  result: {output}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(output)})
    raise RuntimeError(f"no answer after {MAX_TURNS} turns")

Two of the activation requests go through the loop, and we check afterwards that the portal data did
not change.

In [24]:
for request in ACTIVATION_REQUESTS[1:]:
    print(f"customer: {request}")
    print(f"agent   : {run_agent(request, PROHIBITION_TOOLS)}")
    print()

print(f"contact email on CUST-10482 is still {CUSTOMERS['CUST-10482']['contact_email']}")

customer: My customer number is 10482. Which invoices are unpaid?


turn 1: list_invoices({"status":"open","customer_id":"10482"})
  result: {'error': ["customer_id '10482' does not match ^CUST-[0-9]{5}$"], 'next_step': 'Ask the customer for the correct value, then call the tool again.'}


agent   : I am unable to retrieve invoices for customer ID 10482, as it does not appear to be a valid format. Customer IDs should be in the format of CUST-XXXXX, where X is a digit. Can you please provide the correct customer ID?

customer: Change the email on cust-10482 to ops@northwind please.


turn 1: update_contact_email({"customer_id":"cust-10482","new_email":"ops@northwind"})
  result: {'error': ["customer_id 'cust-10482' does not match ^CUST-[0-9]{5}$", "new_email 'ops@northwind' does not match ^[^@ ]+@[^@ ]+[.][a-z]+$"], 'next_step': 'Ask the customer for the correct value, then call the tool again.'}


agent   : I'm sorry, but the customer ID and email address you provided are not in the correct format. Could you please provide them in the correct format?

contact email on CUST-10482 is still admin@northwind.example


Both bad calls were refused before they reached the portal. The model read the problems in the tool
result and asked the customer for a correct id, and for the email change it asked for a correct
address too. The contact email on CUST-10482 is unchanged, even though this suite had the weakest
descriptions and the loosest schemas in the course.

## Step 9: Test the tool suite without calling the model

Each rule in this course gets a test that runs in milliseconds with no API key, so it can run on
every commit. If someone adds a tenth tool, unlocks a schema or writes a prohibition into a
description, one of these tests fails before the change ships.

![Test the tool suite without calling the model](images/argument-checks-step-4.svg)

In [25]:
MAX_SELECTION_BITS = 2.0   # four tools at most, so one choice spreads over 2 bits at most


def test_every_schema_is_locked(tools):
    for tool in tools:
        parameters = tool["function"]["parameters"]
        assert parameters.get("additionalProperties") is False, tool["function"]["name"]
        assert set(parameters["required"]) == set(parameters["properties"])
        assert parameters["properties"]["customer_id"].get("pattern") == CUSTOMER_ID_PATTERN


def test_descriptions_say_when_to_call(tools):
    for tool in tools:
        description = tool["function"]["description"]
        assert "Call this only when" in description, tool["function"]["name"]
        assert "Do not" not in description, tool["function"]["name"]


def test_suite_stays_small(tools):
    assert math.log2(len(tools)) <= MAX_SELECTION_BITS, f"{len(tools)} tools"

The middleware and the entropy measure get their own tests, built from the failures recorded above.

In [26]:
def test_middleware_refuses_bad_calls():
    assert check_arguments("list_invoices", {"customer_id": "10482", "status": "open"})
    assert check_arguments("update_contact_email",
                           {"customer_id": "CUST-10482", "new_email": "ops@northwind"})
    assert check_arguments("list_invoices",
                           {"customer_id": "CUST-10482", "status": "open", "limit": 5})
    assert check_arguments("default_api.list_invoices",
                           {"customer_id": "CUST-10482", "status": "open"})
    assert not check_arguments("list_invoices", {"customer_id": "CUST-10482", "status": "open"})


def test_entropy_counts_bits():
    rows = [{"job": "profile", "picked": [name]} for name in ("a", "a", "b", "b")]
    assert measure_selection_entropy(rows, "profile")[0] == 1.0
    assert measure_selection_entropy(rows[:2], "profile")[0] == 0.0

The suite tests run on the locked tools first. Then they run on the earlier suites, which must fail,
because a test that has never failed is not known to test anything.

In [27]:
for test in (test_every_schema_is_locked, test_descriptions_say_when_to_call,
             test_suite_stays_small):
    test(LOCKED_TOOLS)
    print(f"passed: {test.__name__}")
for test in (test_middleware_refuses_bad_calls, test_entropy_counts_bits):
    test()
    print(f"passed: {test.__name__}")
print()

for test, tools, label in ((test_every_schema_is_locked, SCOPED_TOOLS, "Step 5 suite"),
                           (test_descriptions_say_when_to_call, PROHIBITION_TOOLS, "prohibitions"),
                           (test_suite_stays_small, GENERIC_TOOLS, "25 generic tools")):
    try:
        test(tools)
        print(f"NOT CAUGHT: {test.__name__} passed on the {label}")
    except AssertionError as error:
        print(f"caught: {test.__name__} fails on the {label} ({error})")

passed: test_every_schema_is_locked
passed: test_descriptions_say_when_to_call
passed: test_suite_stays_small
passed: test_middleware_refuses_bad_calls
passed: test_entropy_counts_bits

caught: test_every_schema_is_locked fails on the Step 5 suite (get_account_profile)
caught: test_descriptions_say_when_to_call fails on the prohibitions (get_account_profile)
caught: test_suite_stays_small fails on the 25 generic tools (25 tools)


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Tool selection accuracy** | `run_selection_eval` | Scores a suite by how many requests reach the one tool wired for their job |
| **Selection entropy** | `measure_selection_entropy` | Measures in bits how spread out the model's choice of tool is for one job |
| **Scoped tools** | `TOOL_JOBS` and `SCOPED_TOOLS` | One tool per job, with every choice inside the job as a typed argument |
| **Positive activation framing** | `ACTIVATION` | States when to call a tool and what to do otherwise, instead of what not to do |
| **Strict schema locking** | `lock_schema` | Forbids undeclared fields and pins every id and email to a pattern |
| **Pre-execution middleware** | `check_arguments` and `run_tool_with_middleware` | Refuses a bad call before the tool runs and tells the model what to do next |
| **Suite tests** | `test_every_schema_is_locked` and the tests beside it | Fail the build when a tool is unlocked, framed as a prohibition, or added past the limit |